## mT5 Training
Use the existing text-to-SPARQL trainer. Download and fine-tuning require explicit switches and model dependencies.

In [ ]:
from pathlib import Path
from uuid import uuid4
import json
import os

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/hlo_recommender").is_dir())
os.chdir(ROOT)

from hlo_recommender.cli import parser, run
from hlo_recommender.io import read_jsonl, write_json, write_jsonl

SESSION_FILE = ROOT / "artifacts/notebooks/session.json"
SESSION = json.loads(SESSION_FILE.read_text()) if SESSION_FILE.exists() else {}


In [ ]:
import subprocess
import sys

INSTALL_MODEL_DEPENDENCIES = False
DOWNLOAD_BASE = False
TRAIN_MODEL = False
if INSTALL_MODEL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(ROOT) + "[model]"], check=True)
BASE = ROOT / "artifacts/base/mt5-small"
if DOWNLOAD_BASE and not (BASE / "base_manifest.json").is_file():
    run(parser().parse_args(["download", "--model", "google/mt5-small", "--output", str(BASE)]))
{"base_available": (BASE / "base_manifest.json").is_file(), "training_enabled": TRAIN_MODEL}


### Configuration
Reuse the development configuration and isolated run outputs; reviewed data requires reviewed partitions and labels.

In [ ]:
OUTPUT = ROOT / "artifacts/notebooks" / f"mt5-{uuid4().hex[:8]}"
CONFIG_SOURCE = "configs/mt5.development.toml"
PATH_FIELDS = ("runtime", "train", "validation", "base_checkpoint")
TRAIN_FILE = "questions.train.jsonl"
VALIDATION_FILE = "questions.validation.jsonl"

import tomllib

settings = tomllib.loads((ROOT / CONFIG_SOURCE).read_text())
for key in PATH_FIELDS:
    settings[key] = str(ROOT / settings[key])
DATA = Path(SESSION.get("data_directory", ROOT / "data/development"))
settings["train"] = str(DATA / TRAIN_FILE)
settings["validation"] = str(DATA / VALIDATION_FILE)
settings["output"] = str(OUTPUT)
CONFIG_PATH = ROOT / "artifacts/notebooks/configs" / f"{OUTPUT.name}.toml"
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
CONFIG_PATH.write_text("\n".join(f"{key} = {json.dumps(value)}" for key, value in settings.items()) + "\n")
settings


### Preflight
Validate the paired questions and SPARQL targets before loading neural weights.

In [ ]:
preflight = run(parser().parse_args(["train", "mt5", "--config", str(CONFIG_PATH), "--allow-synthetic", "--check-only"]))
preflight


### Fine-tuning
The project trainer uses AdamW, warmup, clipping, execution-based validation and early stopping.

In [ ]:
if TRAIN_MODEL:
    training = run(parser().parse_args(["train", "mt5", "--config", str(CONFIG_PATH), "--allow-synthetic"]))
    SESSION["mt5_checkpoint"] = training["checkpoint"]
    write_json(SESSION_FILE, SESSION)
    print(json.dumps(training, indent=2))
else:
    print("Preflight completed; mT5 fine-tuning was not executed.")


In [ ]:
if (OUTPUT / "history.json").is_file():
    history = json.loads((OUTPUT / "history.json").read_text())
    print(json.dumps(history, indent=2))
